#  Control de Sobreajuste (Overfitting) en Redes Neuronales
## Módulo 6 · Día 2 · Dropout & Early Stopping · UDLA

**Instructor: David Ponce**

---

###  ¿Qué aprenderemos hoy?

Ayer construimos una red neuronal que predecía porosidad. Hoy enfrentamos el
**problema más importante del deep learning: el OVERFITTING** — cuando la red
memoriza en vez de aprender.

Aprenderemos a:
1. **Forzar** el overfitting a propósito para reconocerlo
2. **Detectarlo** mirando las curvas de pérdida
3. **Corregirlo** con Dropout y Early Stopping
4. **Validar** que la red generaliza a datos nuevos

>  **Tip:** Lee los comentarios línea por línea. Este día es 50% teoría
> y 50% ver el overfitting en acción con tus propios ojos.

---
##  PARTE 1: Importando las herramientas

Hoy agregamos DOS herramientas nuevas a las de ayer:
- `Dropout`: capa que apaga neuronas aleatoriamente
- `EarlyStopping`: callback que detiene el entrenamiento a tiempo

###  Celda 1: Importación de librerías

In [ ]:
# ============================================
# CELDA 1: Importación de librerías
# ============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
#    'Dropout' = NUEVA capa. Apaga aleatoriamente un % de neuronas en cada
#      paso de entrenamiento, forzando a la red a ser robusta.
from tensorflow.keras.callbacks import EarlyStopping
#    'EarlyStopping' = NUEVO callback. Detiene el entrenamiento cuando la
#      pérdida de validación deja de mejorar, evitando el sobreajuste.

sns.set_theme(style="whitegrid")
print(" Librerías importadas.")


###  Las 2 herramientas nuevas

| Herramienta | ¿Qué hace? | Analogía |
|-------------|------------|----------|
| `Dropout(0.2)` | Apaga 20% de neuronas al azar en cada paso | Vendar ojos al 20% del equipo de fútbol en cada práctica |
| `EarlyStopping` | Detiene si no mejora en N épocas | El entrenador que dice 'ya es suficiente' |


###  Celda 2: Cargar y preparar los datos (igual que ayer)

In [ ]:
!wget -q https://raw.githubusercontent.com/DavidPonce84/machine-learning-course/main/modulo_6_deep_learning/data/registro_petrofisico.csv -O registro_petrofisico.csv
# ============================================
# CELDA 2: Cargar datos, escalar y dividir
# ============================================

df_well = pd.read_csv('registro_petrofisico.csv')
#    Carga el dataset de well logs.

features = ['GR_API', 'ILD_ohm_m', 'RHOB_g_cc']
target = ['NPHI_v_v']

# ─── Escalar entradas y salida ───
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(df_well[features])
y_scaled = scaler_y.fit_transform(df_well[target])
#    Escalamos TODO (entradas y salida), igual que ayer.

# ─── Dividir en entrenamiento (80%) y validación (20%) ───
profundidad = df_well['Profundidad_m'].values
X_train, X_val, y_train, y_val, prof_train, prof_val = train_test_split(
    X_scaled, y_scaled, profundidad, test_size=0.2, random_state=42
)
print(f" Entrenamiento: {X_train.shape[0]} · Validación: {X_val.shape[0]}")


---
##  PARTE 3: Forzando el Overfitting a Propósito

Para entender al enemigo, primero lo creamos. Vamos a entrenar una red
**sobredimensionada** (muchas neuronas) sin regularización y por **muchas
épocas** (100). El resultado: la red memorizará el ruido del pozo.


###  Celda 3: Modelo sobredimensionado SIN regularización

In [ ]:
# ============================================
# CELDA 3: Forzar overfitting — red grande sin regularización
# ============================================

model_over = Sequential([
    Dense(128, activation='relu', input_shape=(3,)),
    #    128 neuronas (más de lo necesario -> más capacidad de memorizar).
    Dense(128, activation='relu'),
    #    Otra capa de 128. Mucha capacidad.
    Dense(1, activation='linear')
    #    Salida para regresión.
])
#    NOTA: no hay Dropout. Esta red puede memorizar libremente.

model_over.compile(optimizer='adam', loss='mse')
#    Compilamos igual que ayer.

history_over = model_over.fit(
    X_train, y_train,
    epochs=100,              # ← 100 épocas (muchas -> más chance de memorizar)
    batch_size=16,           # ← batches pequeños -> actualizaciones más frecuentes
    validation_data=(X_val, y_val),
    verbose=0                # ← silencioso (no imprimir cada época)
)
#    Entrenamos SIN EarlyStopping -> la red corre las 100 épocas completas.
print(" Modelo sobredimensionado entrenado por 100 épocas.")


###  Celda 4: Visualizar el overfitting

In [ ]:
# ============================================
# CELDA 4: Curva de pérdida — la firma del overfitting
# ============================================

plt.figure(figsize=(10, 5))
plt.plot(history_over.history['loss'], label='Entrenamiento', linewidth=2)
plt.plot(history_over.history['val_loss'], label='Validación', linewidth=2)
plt.title('Divergencia de Pérdidas: Overfitting en Acción')
plt.xlabel('Época')
plt.ylabel('MSE')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
#    La FIRMA del overfitting: 'loss' (entrenamiento) sigue bajando,
#      pero 'val_loss' (validación) empieza a SUBIR o a separarse.
#      La red dejó de aprender y empezó a memorizar.


###  La firma del overfitting

```
MSE
  
  |        val_loss  (sube)
  |       /
  |      /  ← el punto de separación
  |     /
  |    loss  (sigue bajando)
  └─────────────────-> época
```
Cuando la línea de validación empieza a subir mientras la de entrenamiento
sigue bajando, la red está **memorizando** en vez de **aprender**.


---
##  PARTE 4: Corrigiendo con Dropout + Early Stopping

Ahora construimos la MISMA red pero con dos defensas:
1. **Dropout(0.2)**: apaga 20% de neuronas en cada paso
2. **EarlyStopping**: detiene si no mejora en 10 épocas


###  Celda 5: Modelo con Dropout y Early Stopping

In [ ]:
# ============================================
# CELDA 5: Red regularizada con Dropout
# ============================================

model_reg = Sequential([
    Dense(128, activation='relu', input_shape=(3,)),
    Dropout(0.2),
    #    'Dropout(0.2)' = en cada paso, apaga aleatoriamente el 20% de las
    #      neuronas de la capa anterior. La red no puede depender de una
    #      sola neurona "estrella".
    Dense(128, activation='relu'),
    Dropout(0.2),
    #    Segundo Dropout después de la segunda capa oculta.
    Dense(1, activation='linear')
])
model_reg.compile(optimizer='adam', loss='mse')

# ─── Configurar Early Stopping ───
early_stop = EarlyStopping(
    monitor='val_loss',        # ← observa la pérdida de VALIDACIÓN
    patience=10,               # ← si no mejora en 10 épocas seguidas, PARA
    restore_best_weights=True  # ← restaura los pesos de la mejor época
)
#    'EarlyStopping' es un callback: se ejecuta al final de cada época.
#    Si val_loss no mejora durante 'patience' épocas, detiene el training.
#    'restore_best_weights' = vuelve a los pesos del mejor momento,
#      no deja la red en el estado final (que podría estar peor).
print(" Modelo regularizado listo con Early Stopping.")


###  Celda 6: Entrenar el modelo regularizado

In [ ]:
# ============================================
# CELDA 6: Entrenar con Early Stopping
# ============================================

history_reg = model_reg.fit(
    X_train, y_train,
    epochs=100,                  # ← dejamos 100 épocas máximas...
    batch_size=16,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],      # ← ...pero EarlyStopping puede cortar antes
    verbose=0
)
#    'callbacks=[early_stop]' = activa la parada temprana.
#    La red entrenará hasta que val_loss deje de mejorar por 10 épocas.
print(f" Entrenamiento detenido en la época {len(history_reg.history['loss'])}.")
print(f"   (Early Stopping cortó en vez de correr las 100 épocas completas)")


###  Celda 7: Comparar las curvas regularizada vs sobreajustada

In [ ]:
# ============================================
# CELDA 7: Comparación lado a lado
# ============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
#    Dos gráficos lado a lado para comparar.

# --- Izquierda: sin regularización ---
axes[0].plot(history_over.history['loss'], label='Entrenamiento', linewidth=2)
axes[0].plot(history_over.history['val_loss'], label='Validación', linewidth=2)
axes[0].set_title('Sin Regularización (Overfitting)')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('MSE')
axes[0].legend()

# --- Derecha: con regularización ---
axes[1].plot(history_reg.history['loss'], label='Entrenamiento', linewidth=2)
axes[1].plot(history_reg.history['val_loss'], label='Validación', linewidth=2)
axes[1].set_title('Con Dropout + Early Stopping')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('MSE')
axes[1].legend()

plt.tight_layout()
plt.show()
#    Izquierda: validación sube (memorizó).
#    Derecha: ambas bajan y se estabilizan (aprendió).


###  La diferencia clave

| | Sin regularización | Con Dropout + Early Stopping |
|---|---|---|
| Pérdida entrenamiento | Muy baja (memorizó) | Baja (aprendió) |
| Pérdida validación | Sube | Estable y baja |
| Generalización | Pésima | Excelente |

>  **El objetivo NO es el error más bajo en entrenamiento.** Es el error
> más bajo en datos NUEVOS que la red nunca vio.

---
##  PARTE 5: Validando en el tramo ciego

La prueba final: predecir en los datos de validación (que la red nunca vio)
y comparar con la porosidad real.

###  Celda 8: Predicción final en el pozo ciego

In [ ]:
# ============================================
# CELDA 8: Validar generalización en el tramo ciego
# ============================================

# ─── Predecir con el modelo regularizado ───
y_pred_scaled = model_reg.predict(X_val)
y_pred = scaler_y.inverse_transform(y_pred_scaled)
#    Predice y des-escala (igual que ayer).

y_val_real = scaler_y.inverse_transform(y_val)
#    Valores reales des-escalados para comparar.

# ─── La profundidad de validación ya fue guardada en la Celda 2 ───
# prof_val contiene la profundidad de los datos de validación.

plt.figure(figsize=(10, 6))
plt.scatter(prof_val, y_val_real, s=3, alpha=0.5, label='NPHI Real', color='#38bdf8')
plt.scatter(prof_val, y_pred, s=3, alpha=0.5, label='NPHI Predicha', color='#f59e0b')
plt.xlabel('Profundidad (m)')
plt.ylabel('Porosidad NPHI (v/v)')
plt.title('Validación Final: Porosidad Real vs Predicha (Tramo Ciego)')
plt.legend()
plt.show()
#    Si azul y naranja se solapan en el tramo CIEGO -> la red generalizó.
#      No solo memorizó el 80% de entrenamiento.


---
##  RECAP: El Pipeline de Regularización

```
┌────────────────────────────────────────────────────────┐
│ 1. SOBREAJUSTAR (a propósito) -> red grande, 100 épocas │
│ 2. DETECTAR -> curva validación sube mientras entrena baja│
│ 3. CORREGIR -> Dropout(0.2) + EarlyStopping(patience=10) │
│ 4. COMPARAR -> lado a lado: memorizar vs aprender        │
│ 5. VALIDAR -> tramo ciego: ¿generaliza?                 │
└────────────────────────────────────────────────────────┘
```

###  Lo que aprendiste hoy

- Overfitting = memorizar en vez de aprender
- La firma: entrenamiento baja, validación sube
- Dropout: apaga neuronas aleatoriamente para robustecer
- Early Stopping: detiene a tiempo
- El tramo ciego es el juez final

>  **Fin del Módulo 6.** Has construido, entrenado y regularizado una red
> neuronal para predecir porosidad. El último módulo (7) es el proyecto integrador.


---
## PARTE 6: Laboratorio Interactivo — Dropout y Neuronas

Ahora explora el efecto de Dropout y del número de neuronas. Mueve los sliders
y observa cómo cambia la curva de pérdida.

**En lenguaje simple:**
- Dropout = 0.0: sin defensa, la red memoriza (overfitting).
- Dropout = 0.3-0.5: buena regularización.
- Dropout = 0.8: apaga demasiadas neuronas, la red no aprende.


In [ ]:
# ============================================
# CELDA 9 (INTERACTIVA): Laboratorio de Dropout
# ============================================

import ipywidgets as widgets
from IPython.display import clear_output

def laboratorio(neuronas=64, dropout=0.2, epochs=60):
    clear_output(wait=True)
    model = Sequential([
        Dense(neuronas, activation='relu', input_shape=(3,)),
        Dropout(dropout),
        Dense(neuronas // 2, activation='relu'),
        Dropout(dropout),
        Dense(1, activation='linear')
    ])
    model.compile(optimizer='adam', loss='mse')
    history = model.fit(X_train, y_train, epochs=epochs,
                        validation_data=(X_val, y_val), verbose=0)
    plt.figure(figsize=(10, 4))
    plt.plot(history.history['loss'], label='Entrenamiento')
    plt.plot(history.history['val_loss'], label='Validación')
    plt.title(f'Dropout={dropout}, {neuronas} neuronas')
    plt.xlabel('Época'); plt.ylabel('MSE')
    plt.legend(); plt.show()
    print(f'Pérdida final validación: {history.history["val_loss"][-1]:.4f}')

widgets.interact(laboratorio,
                 neuronas=widgets.IntSlider(8, 128, step=8, value=64),
                 dropout=widgets.FloatSlider(0.0, 0.8, step=0.1, value=0.2),
                 epochs=widgets.IntSlider(20, 150, step=10, value=60))
#   Sube dropout a 0.8: la red deja de aprender (apaga demasiado).
#   Baja dropout a 0.0: la red memoriza (overfitting).
